In [26]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# Корень проекта (на уровень выше папки notebooks)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
print(DATA, DATA.exists())

/home/mchunikhin/project_steam_game_recomm_system/data True


## Reviews → interactions / Отзывы → взаимодействия

The raw data is huge: ~125k games and ~15.4M reviews (2010–2021), about 5.7 GB of CSV.
Most games have very few reviews, so we set thresholds to get a clean, workable dataset:

- **Time window:** keep reviews from 2020-01-01 to 2021-04-23 (the freshest ~1 year 4 months).
- **Games:** keep only games with ≥ 20 reviews in this window.
- **Users:** keep only users with ≥ 5 interactions (needed for the leave-last-out split).

Thresholds are applied repeatedly until the sets stop changing (k-core).
Result: ~2.9k games, ~114k users, ~901k interactions → `interactions.parquet`.

Сырых данных очень много: ~125k игр и ~15.4M отзывов (2010–2021), около 5.7 ГБ CSV.
У большинства игр отзывов почти нет, поэтому ставим пороги для чистого рабочего датасета:

- **Период:** отзывы с 2020-01-01 по 2021-04-23 (самые свежие ~1 год 4 мес).
- **Игры:** оставляем игры с ≥ 20 отзывов в этом окне.
- **Пользователи:** оставляем юзеров с ≥ 5 взаимодействиями (нужно для split leave-last-out).

Пороги применяем по очереди, пока множества не перестанут меняться (k-core).
Итог: ~2.9k игр, ~114k юзеров, ~901k взаимодействий → `interactions.parquet`.

In [ ]:
REBUILD = False  # True — пересобрать заново, игнорируя кэш

# Параметры фильтрации (зафиксированы для проекта)
WINDOW_START = int(datetime(2020, 1, 1, tzinfo=timezone.utc).timestamp())  # 2020-01-01
MIN_GAME_REVIEWS = 20   # игра остаётся, если у неё >= 20 отзывов в окне
MIN_USER_INTER   = 5    # юзер остаётся, если у него >= 5 взаимодействий
REVIEW_COLS = ["steamid", "appid", "voted_up", "playtime_forever",
               "unix_timestamp_created"]

out = DATA / "processed" / "interactions.parquet"
if out.exists() and not REBUILD:
    inter = pd.read_parquet(out)
    print("загружено из кэша:", out, f"({len(inter):,} строк)")
else:
    # Читаем отзывы чанками, оставляем только нужное окно и колонки
    parts = []
    for f in sorted((DATA / "reviews").glob("reviews-*.csv")):
        for ch in pd.read_csv(f, usecols=REVIEW_COLS, chunksize=3_000_000):
            ch = ch[ch["unix_timestamp_created"] >= WINDOW_START]
            if len(ch):
                parts.append(ch)
    inter = pd.concat(parts, ignore_index=True)
    del parts
    print("в окне:", f"{len(inter):,}")

    # Дедуп: один юзер на одну игру, оставляем самый свежий отзыв
    inter = (inter.sort_values("unix_timestamp_created")
                  .drop_duplicates(["steamid", "appid"], keep="last"))
    print("после дедупа:", f"{len(inter):,}")

    # Итеративная фильтрация по порогам (k-core), пока множества не стабилизируются
    while True:
        n0 = len(inter)
        g_ok = inter["appid"].value_counts()
        inter = inter[inter["appid"].isin(g_ok[g_ok >= MIN_GAME_REVIEWS].index)]
        u_ok = inter["steamid"].value_counts()
        inter = inter[inter["steamid"].isin(u_ok[u_ok >= MIN_USER_INTER].index)]
        if len(inter) == n0:
            break

    # Итоговая таблица взаимодействий
    inter = inter.rename(columns={"steamid": "user_id", "appid": "game_id",
                                  "voted_up": "target"})
    inter["target"] = inter["target"].astype("int8")
    inter.to_parquet(out, index=False)
    print("сохранено:", out, f"({out.stat().st_size/1e6:.1f} MB)")

print("\nИТОГ")
print("взаимодействий:", f"{len(inter):,}")
print("игр:          ", f"{inter['game_id'].nunique():,}")
print("юзеров:       ", f"{inter['user_id'].nunique():,}")

## Games metadata / Метаданные игр

We load game metadata (useful columns only), keep only the games present in `interactions`,
and de-duplicate by `AppID`. This is the item-features table.

Note: `index_col=False` is required — rows have one extra field vs the header, otherwise
pandas treats `AppID` as the index and all columns shift.
Result: ~2.8k games → `games.parquet`.

Загружаем метаданные игр (только нужные колонки), оставляем лишь игры из `interactions`
и дедуплицируем по `AppID`. Это таблица фичей для item-tower.

Важно: нужен `index_col=False` — в строках на одно поле больше, чем в заголовке, иначе
pandas берёт `AppID` за индекс и все колонки съезжают.
Итог: ~2.8k игр → `games.parquet`.

In [ ]:
GAME_COLS = ["AppID", "Name", "Release date", "Estimated owners", "Price",
             "Positive", "Negative", "Genres", "Tags", "Categories",
             "Average playtime forever", "Median playtime forever"]

out_g = DATA / "processed" / "games.parquet"
if out_g.exists() and not REBUILD:
    games_f = pd.read_parquet(out_g)
    print("загружено из кэша:", out_g, f"({len(games_f):,} игр)")
else:
    # games.csv ~400 МБ. index_col=False обязателен (см. markdown).
    games = pd.read_csv(DATA / "games" / "games.csv", usecols=GAME_COLS, index_col=False)

    # Оставляем только игры из interactions, дедуп по AppID
    keep_ids = inter["game_id"].unique()
    games_f = (games.drop_duplicates("AppID")
               .loc[lambda d: d["AppID"].isin(keep_ids)]
               .rename(columns={"AppID": "game_id"})
               .reset_index(drop=True))
    games_f.to_parquet(out_g, index=False)
    print("игр в interactions:", f"{len(keep_ids):,}")
    print("без метаданных:     ", f"{len(set(keep_ids) - set(games_f['game_id'])):,}")
    print("сохранено:", out_g, f"({out_g.stat().st_size/1e6:.1f} MB)")

print("игр с метаданными:", f"{len(games_f):,}")